# Uplift Benchmark — Exploration Notebook

Interactive exploration of the DGPs, models, and metrics before running the full benchmark.

**Use this to:**
- Understand the data generating processes visually
- Sanity-check a single model on a single DGP
- Inspect uplift curves and calibration interactively
- Prototype new DGPs or metrics before adding them to the benchmark

In [ ]:
import sys
sys.path.insert(0, '..')  # run from notebooks/ directory

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.1)
%matplotlib inline

## 1. Visualize the DGPs

For each DGP, let's look at the distribution of true CATE values and how confounding shifts the propensity score distribution.

In [ ]:
from data.dgp import LinearDGP, NonlinearDGP, HeterogeneousDGP

dgps = [
    ('Linear CATE', LinearDGP(alpha=1.0, rho=0.05, seed=42)),
    ('Nonlinear CATE', NonlinearDGP(alpha=1.0, rho=0.05, seed=42)),
    ('Heterogeneous CATE', HeterogeneousDGP(alpha=1.0, rho=0.05, seed=42)),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, dgp) in zip(axes, dgps):
    sample = dgp.sample(10000)
    ax.hist(sample.tau, bins=50, edgecolor='none', color='steelblue', alpha=0.8)
    ax.axvline(sample.tau.mean(), color='red', linestyle='--', linewidth=2,
               label=f'ATE = {sample.tau.mean():.3f}')
    ax.axvline(0, color='gray', linestyle=':', linewidth=1)
    ax.set_title(name)
    ax.set_xlabel('True CATE τ(X)')
    ax.set_ylabel('Count')
    ax.legend()

fig.suptitle('Distribution of True CATE across DGPs', fontsize=13)
plt.tight_layout()
plt.show()

## 2. Effect of Confounding on Propensity

As α increases, treated and control users have increasingly different covariate distributions.

In [ ]:
from data.dgp import HeterogeneousDGP

alphas = [0.0, 1.0, 2.0]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, alpha in zip(axes, alphas):
    dgp = HeterogeneousDGP(alpha=alpha, rho=0.05, seed=42)
    sample = dgp.sample(5000)
    
    ax.hist(sample.propensity[sample.T == 1], bins=30, alpha=0.6, 
            label='Treated', color='steelblue', density=True)
    ax.hist(sample.propensity[sample.T == 0], bins=30, alpha=0.6,
            label='Control', color='coral', density=True)
    ax.set_title(f'α = {alpha}')
    ax.set_xlabel('Propensity P(T=1|X)')
    ax.set_ylabel('Density')
    ax.legend()

fig.suptitle('Propensity Score Overlap by Confounding Level', fontsize=13)
plt.tight_layout()
plt.show()

## 3. The Sure-Thing Problem

Visualize what happens when a model optimizes for conversion probability vs. true CATE.

In [ ]:
from data.dgp import HeterogeneousDGP
from sklearn.ensemble import RandomForestClassifier
import numpy as np

dgp = HeterogeneousDGP(alpha=1.5, rho=0.05, seed=42)
sample = dgp.sample(20000)
train, test = sample.train_test_split(test_size=0.3, seed=42)

# Naive model: predict P(convert) — ignores incrementality
naive_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
naive_model.fit(train.X, train.Y)
naive_score = naive_model.predict_proba(test.X)[:, 1]

# True CATE ranking
true_cate = test.tau

# Show: high predicted P(convert) ≠ high CATE
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(naive_score, true_cate, alpha=0.1, s=5, c='steelblue')
axes[0].axhline(0, color='red', linestyle='--', linewidth=1, label='τ=0 (sure-thing boundary)')
axes[0].set_xlabel('Naive model: P(convert | X)')
axes[0].set_ylabel('True CATE τ(X)')
axes[0].set_title('Conversion Probability vs. True Incrementality')
axes[0].legend()

# Wasted spend under naive policy
top_k = 0.30
n = len(test.Y)
k = int(top_k * n)
naive_order = np.argsort(-naive_score)  # sort by predicted P(convert)
true_order = np.argsort(-true_cate)     # sort by true CATE

naive_top_cate = true_cate[naive_order[:k]]
oracle_top_cate = true_cate[true_order[:k]]

axes[1].hist(naive_top_cate, bins=40, alpha=0.7, label=f'Naive policy\n(sort by P(convert))', 
             color='coral', density=True)
axes[1].hist(oracle_top_cate, bins=40, alpha=0.7, label='Oracle policy\n(sort by true CATE)', 
             color='steelblue', density=True)
axes[1].axvline(0, color='black', linestyle='--', linewidth=1)
axes[1].set_xlabel('True CATE of targeted users')
axes[1].set_ylabel('Density')
axes[1].set_title(f'True CATE Distribution: Top {int(top_k*100)}% Targeted')
axes[1].legend()

naive_wasted = (naive_top_cate < 0.01).mean()
oracle_wasted = (oracle_top_cate < 0.01).mean()
print(f'Naive policy wasted spend:  {naive_wasted*100:.1f}%')
print(f'Oracle policy wasted spend: {oracle_wasted*100:.1f}%')

plt.tight_layout()
plt.show()

## 4. Run a Single Model End-to-End

In [ ]:
from models import SLearner, TLearner, XLearner, RLearner, UpliftRF
from metrics import evaluate, uplift_curve
from data.dgp import HeterogeneousDGP

dgp = HeterogeneousDGP(alpha=1.0, rho=0.05, seed=0)
sample = dgp.sample(20000)
train, test = sample.train_test_split(test_size=0.3, seed=0)

models_to_run = [
    ('S-Learner [RF]', SLearner(learner_type='rf', seed=0)),
    ('T-Learner [RF]', TLearner(learner_type='rf', seed=0)),
    ('X-Learner [RF]', XLearner(learner_type='rf', seed=0)),
    ('R-Learner [RF]', RLearner(learner_type='rf', seed=0)),
    ('Uplift RF',      UpliftRF(seed=0)),
]

results = []
for name, model in models_to_run:
    print(f'Fitting {name}...')
    model.fit(train.X, train.T, train.Y)
    tau_pred = model.predict_cate(test.X)
    metrics = evaluate(tau_pred, test.Y, test.T, tau_true=test.tau, top_k=0.30)
    results.append({'model': name, **metrics})

results_df = pd.DataFrame(results).set_index('model')
display_cols = ['pehe', 'qini', 'wasted_spend', 'iroas', 'oracle_iroas', 'iroas_efficiency']
print('\n--- Results ---')
display(results_df[display_cols].round(4))

## 5. Uplift Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = plt.cm.tab10.colors

for (name, model), color in zip(models_to_run, colors):
    tau_pred = model.predict_cate(test.X)
    phi, lift = uplift_curve(tau_pred, test.Y, test.T)
    ax.plot(phi * 100, lift, label=name, linewidth=1.5, color=color)

# Oracle
phi_oracle, lift_oracle = uplift_curve(test.tau, test.Y, test.T)
ax.plot(phi_oracle * 100, lift_oracle, 'k--', linewidth=2, label='Oracle')
ax.plot([0, 100], [0, lift_oracle[-1]], ':', color='gray', linewidth=1, label='Random')

ax.set_xlabel('Population treated (%)')
ax.set_ylabel('Cumulative incremental conversions')
ax.set_title('Uplift Curves — Heterogeneous DGP, α=1.0')
ax.legend(fontsize=9)
ax.set_xlim(0, 100)
plt.tight_layout()
plt.show()